# Synthetic Personal-Assistant Data: Exploratory Data Analysis

This notebook analyzes the completed message-render output `data/lifecycle_20261021_1318/02_rendered/messages/events_with_messages_rendered.jsonl`. It reads event rows from that file and persona/configuration metadata from the immutable `01_raw_generated/` phase. It never edits either input and writes a new, separate analysis run to `outputs/eda_rendered/` so the earlier raw-phase EDA artifacts remain intact.

The benchmark represents 30-day simulated traces. Its intended downstream system covers event retrieval, planning, preference learning, bounded actions, deletion, privacy, and safety. Event fields include event/persona IDs, day and minute, event type, payload, optional fact/preference attributes, sensitivity, provenance, checker labels, text, and render status. Model-specific train/validation examples are created only after trace/label processing; rendered events are not themselves a training export.

## Research-grounded analysis choices

| Technique | Why / issue detected | Metric or test | Evidence and limitation |
|---|---|---|---|
| Multi-axis quality and privacy assessment | Prevents one fidelity score from standing in for utility, quality, and privacy | Integrity, coverage, missingness, lexical uniqueness, privacy screen reported separately | Lautrup et al., [Syntheval](https://doi.org/10.1007/s10618-024-01081-4); designed for tabular synthetic data, so text and sequences need adaptation. |
| Text vs structured-label validation | Rendered text can contradict its structured facts/labels | Render status, exact duplicate text, later label-integrity gates | Ramesh et al., [SynthTextEval](https://aclanthology.org/2025.emnlp-demos.35/); automatic measures do not prove naturalness or cultural authenticity; human review is still needed. |
| Subgroup representation and fairness cautions | Exposes absent or underrepresented generated cohorts without presuming equal preferences | Persona counts by split, archetype, locale, region, name-list proxy; outcome parity is not asserted without a defined outcome | Nagesh et al., [FairCauseSyn](https://doi.org/10.1109/EMBC58623.2025.11252705); health-data case study, and causal fairness requires a defensible task-specific causal model. |
| Temporal tail/coverage checks | Detects missing days and changing event mix; preserves rare behaviors from disappearing | Per-day and per-type counts, split-wise Jensen–Shannon divergence | Shumailov et al., [The Curse of Recursion](https://doi.org/10.1038/s41586-024-07566-y); motivates caution about recursive synthetic training, not a claim that any synthetic dataset collapses. |
| Downstream retrieval evaluation | Synthetic quality is ultimately task-dependent | This notebook reports readiness only; the project’s PersonaBench-style retrieval/task harness must be run on held-out personas | Tan et al., [PersonaBench](https://aclanthology.org/2025.findings-acl.49/); benchmark precedent, not a generator-quality certificate. |

All metric values are diagnostics, not universal pass thresholds. Statistical significance alone would not imply practical similarity. Real-world personal communications, a suitable real holdout, and model predictions are not part of the available input.

## 1. Reproducibility and source selection

The analysis uses a fixed seed for any stochastic diagnostic, records package versions and source hashes, and writes a manifest. The rendered events file is selected explicitly; the raw phase is used only for personas, split metadata, and provenance. JSONL, JSON, CSV, and Parquet formats are supported. All source files remain read-only.

In [ ]:
from pathlib import Path
import json
import sys

cwd = Path.cwd().resolve()
if (cwd / "pyproject.toml").exists() and (cwd / "src" / "p17sim").is_dir():
    PROJECT_ROOT = cwd
elif (cwd / "sandbox" / "pyproject.toml").exists():
    PROJECT_ROOT = cwd / "sandbox"
else:
    raise FileNotFoundError("Open/run this notebook from the repository or sandbox directory.")

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from synthetic_data_eda import discover_raw_dir, run_eda

RAW_DIR = discover_raw_dir(PROJECT_ROOT)
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "eda"
print(f"Project: {PROJECT_ROOT}")
print(f"Read-only raw input: {RAW_DIR}")
print(f"Separate output directory: {OUTPUT_DIR}")

## 2. Dataset overview and generation metadata

Summarizes file counts, schemas, data types, cardinalities, memory usage, generation configuration, random seed, and split manifest. Counts are descriptive; they do not establish population representativeness.

In [ ]:
RESULT = run_eda(project_root_override=PROJECT_ROOT, raw_override=RAW_DIR)
print(json.dumps(RESULT, indent=2))
overview = json.loads((OUTPUT_DIR / "statistics" / "dataset_overview.json").read_text())
for item in overview["generated_files"]:
    print(f"{item['file']}: {item['records']:,} rows, {item['features']} fields, {item['memory_bytes']:,} bytes")

## 3. Data quality, schema, and integrity

Reports null/blank values, day and minute bounds, duplicate event IDs and rows, persona references, and split assignments. No rows are dropped or repaired. A missing `text` field for a `pending` event is expected at the raw generation phase, not silently treated as a cleaning action. Refer to `tables/event_missingness.csv` and `tables/integrity_checks.csv`.

In [ ]:
import pandas as pd
display(pd.read_csv(OUTPUT_DIR / "tables" / "dataset_inventory.csv"))
display(pd.read_csv(OUTPUT_DIR / "tables" / "integrity_checks.csv"))
display(pd.read_csv(OUTPUT_DIR / "tables" / "event_missingness.csv").head(20))

## 4. Temporal patterns and event coverage

Per-day counts, hourly event timing, and day-by-event-type counts identify missing days, irregular volume, and temporal shifts. The split-wise event-type Jensen–Shannon divergence is exploratory and sensitive to small cohort sizes; it is not a fidelity score against real users. See the saved time-series figures and daily/split tables.

In [ ]:
from IPython.display import Image, display
for filename in ["records_per_day.png", "daily_event_type_heatmap.png", "events_by_hour.png"]:
    path = OUTPUT_DIR / "figures" / filename
    if path.exists():
        print(filename)
        display(Image(filename=str(path)))
display(pd.read_csv(OUTPUT_DIR / "tables" / "daily_record_counts.csv").head(10))
display(pd.read_csv(OUTPUT_DIR / "tables" / "adjacent_split_event_type_js.csv"))

## 5. Reference fidelity, subgroup representation, and fairness

No real personal-data comparison set is bundled in the raw phase (the generation manifest reports no external personal-data input). Therefore real-vs-synthetic Wasserstein/KS/JS comparisons, equalized-odds metrics, and real-population representation claims are unavailable and must not be fabricated. Synthetic archetype, locale, region, renderer, and name-list fields are proxies that describe the generator design, not demographic truth. Group counts are coverage diagnostics only; demographic parity is not presumed to be the correct objective for personalized preferences.

In [ ]:
subgroup_path = OUTPUT_DIR / "tables" / "persona_subgroup_counts.csv"
display(pd.read_csv(subgroup_path) if subgroup_path.exists() else "No subgroup fields were available.")
print("Real reference comparison: unavailable; do not infer representativeness from these synthetic proxy counts.")

## 6. Diversity, duplication, and mode-collapse indicators

Reports distinct structured values and exact normalized-text repetition. These lexical signals can flag repetition but do not measure semantic diversity, embedding coverage, or mode collapse conclusively. A nearest-neighbor-to-real-data memorization test cannot be run without a suitable real reference dataset. No PCA/UMAP/t-SNE embedding is used because there is no validated feature representation here and such plots would not establish quality.

In [ ]:
diversity = json.loads((OUTPUT_DIR / "statistics" / "diversity_coverage.json").read_text())
print(json.dumps(diversity, indent=2))

## 7. Privacy and memorization-risk screening

The generator uses fictional identities. Regex-like email/phone checks and duplicate-text counts are only screening signals. They are not membership inference, semantic re-identification analysis, or a differential-privacy guarantee. A meaningful membership or nearest-neighbor risk test requires real records that are lawfully available and separately protected; none are supplied here.

In [ ]:
privacy = json.loads((OUTPUT_DIR / "statistics" / "privacy_screening.json").read_text())
print(json.dumps(privacy, indent=2))

## 8. Downstream utility and generalization readiness

This EDA does not train a model or report accuracy. The raw events are not the model-specific exports: rendering/trace validation, label handling, gold workflow checks, and train/validation export must be complete first. Use the project’s persona-separated train and validation inputs; keep test and OOD personas sealed. Report downstream success, deletion compliance, leakage, intervention, preference regret, latency, and cost with uncertainty in the separate benchmark. Real-data generalization is not assessable without a valid held-out real dataset.

## 9. Acceptance criteria and next steps

Structural criteria (valid event IDs, references, day bounds, split assignments) are checkable here. Real-distribution fidelity, model generalization, formal privacy, and model training readiness are explicitly `NOT ASSESSABLE`, `NOT CLAIMED`, or `NOT READY` rather than assigned arbitrary thresholds. If structural failures appear, investigate upstream generation and preserve all rejected samples; do not repair the immutable raw directory.

In [ ]:
display(pd.read_csv(OUTPUT_DIR / "tables" / "acceptance_criteria.csv"))
quality = json.loads((OUTPUT_DIR / "quality_report.json").read_text())
print(f"Raw data modified: {quality['raw_data_modified']}")
print(f"Input file hashes match before/after: {json.loads((OUTPUT_DIR / 'statistics' / 'source_immutability.json').read_text())['hashes_match_before_after']}")
print(f"Full report: {OUTPUT_DIR / 'reports' / 'synthetic_data_eda_report.md'}")